# NexaMem Memory Management Demo

This notebook demonstrates the core memory management capabilities of NexaMem, including:
- Memory storage and retrieval
- Memory scopes and organization
- Channel-based memory separation
- Memory lifecycle operations
- Search and filtering capabilities

## Setup and Imports

First, let's import the necessary modules and set up our environment.

In [ ]:
import sys
import uuid
from datetime import datetime
from pathlib import Path

# Add the src directory to the Python path
project_root = Path.cwd().parent if Path.cwd().name == 'examples' else Path.cwd()
src_path = project_root / 'src'
sys.path.insert(0, str(src_path))

print(f"Project root: {project_root}")
print(f"Source path: {src_path}")
print(f"Python path updated: {str(src_path) in sys.path}")

In [ ]:
# Import NexaMem components
from nexamem.channels import ChannelConfig
from nexamem.config.aimemory import AIMemoryConfig
from nexamem.config.storage import StorageConfig
from nexamem.memory_scope import MemoryScope
from nexamem.new_aimemory import AIMemory

print("✅ NexaMem modules imported successfully")

## Basic Memory Configuration

Let's create a basic in-memory configuration for demonstration purposes.

In [ ]:
# Create a basic storage configuration for in-memory storage
storage_config = StorageConfig(
    type="memory",
    config={
        "database_name": "demo_memory",
        "timeout": 30
    }
)

# Create the AI Memory configuration
config = AIMemoryConfig(
    storage=storage_config,
    channels_yaml=None,  # We'll define channels programmatically
    strict_yaml_validation=False
)

print("✅ Memory configuration created")
print(f"Storage type: {config.storage.type}")
# Database name may be in config.storage.config, depending on implementation
db_name = getattr(config.storage.config, "database_name", None) if hasattr(config.storage, "config") else None
print(f"Database name: {db_name}")

## Initialize Memory System

Now let's create our memory instance and set up some basic channels.

In [ ]:
# Initialize the memory system
memory = AIMemory(config)

# Define some sample channels
channels = {
    "conversation": ChannelConfig(
        name="conversation",
        description="Main conversation history",
        max_entries=1000,
        ttl_sec=60  # No expiration
    ),
    "context": ChannelConfig(
        name="context",
        description="Contextual information and facts",
        max_entries=500,
        ttl_sec=86400  # 24 hours
    ),
    "preferences": ChannelConfig(
        name="preferences",
        description="User preferences and settings",
        max_entries=100,
        ttl_sec=60  # Persistent (no expiration)
    )
}

# Register channels with the memory system
for channel_config in channels.values():
    memory.register_channel(
        name=channel_config.name,
        ttl_sec=channel_config.ttl_sec
    )

print("✅ Memory system initialized")
print(f"Available channels: {list(channels.keys())}")

## Define Memory Scope

Memory scopes help organize memories hierarchically by environment, agent, user, and session.

In [ ]:
# Create a memory scope for our demo
scope = MemoryScope(
    env="demo",
    agent_id="chatbot_assistant",
    user_id="user_12345",
    session_id=str(uuid.uuid4())
)

print("✅ Memory scope created")
print(f"Environment: {scope.env}")
print(f"Agent ID: {scope.agent_id}")
print(f"User ID: {scope.user_id}")
print(f"Session ID: {scope.session_id}")

# Show how key prefixes are generated
for channel_name in channels:
    prefix = scope.get_key_prefix(channel_name)
    print(f"Key prefix for '{channel_name}': {prefix}")

## Storing Memories

Let's store different types of memories in different channels.

In [ ]:
# Store conversation memories
conversation_memories = [
    {
        "role": "user",
        "content": "Hello! I'm looking for investment advice.",
        "timestamp": datetime.now().isoformat()
    },
    {
        "role": "assistant",
        "content": "Hello! I'd be happy to help with investment advice. What's your investment experience level?",
        "timestamp": datetime.now().isoformat()
    },
    {
        "role": "user",
        "content": "I'm a beginner. I want to invest $10,000 for retirement.",
        "timestamp": datetime.now().isoformat()
    }
]

# Store conversation memories
conversation_ids = []
for conv_memory in conversation_memories:
    # Create a scope string from the scope object
    scope_string = f"{scope.env}:{scope.agent_id}:{scope.user_id}:{scope.session_id}"

    memory_id = memory.store(
        content=conv_memory["content"],
        scope=scope_string,
        metadata={
            "role": conv_memory["role"],
            "timestamp": conv_memory["timestamp"],
            "channel": "conversation"  # Include channel in metadata
        }
    )
    conversation_ids.append(memory_id)
    print(f"✅ Stored conversation memory: {memory_id[:8]}...")

print(f"\nStored {len(conversation_ids)} conversation memories")

In [ ]:
# Store contextual information
context_memories = [
    {
        "content": "User is a beginner investor",
        "metadata": {"category": "user_profile", "confidence": 0.9}
    },
    {
        "content": "Investment goal: retirement planning",
        "metadata": {"category": "investment_goal", "confidence": 0.95}
    },
    {
        "content": "Investment amount: $10,000",
        "metadata": {"category": "financial_info", "confidence": 1.0}
    }
]

# Store context memories
context_ids = []
for ctx_memory in context_memories:
    # Create scope string
    scope_string = f"{scope.env}:{scope.agent_id}:{scope.user_id}:{scope.session_id}"

    # Merge metadata with channel information
    metadata = ctx_memory["metadata"].copy()
    metadata["channel"] = "context"

    memory_id = memory.store(
        content=ctx_memory["content"],
        scope=scope_string,
        metadata=metadata
    )
    context_ids.append(memory_id)
    print(f"✅ Stored context memory: {memory_id[:8]}...")

print(f"\nStored {len(context_ids)} context memories")

In [ ]:
# Store user preferences
preference_memories = [
    {
        "content": "Risk tolerance: Conservative",
        "metadata": {"type": "risk_preference", "importance": "high"}
    },
    {
        "content": "Preferred communication style: Simple explanations",
        "metadata": {"type": "communication_style", "importance": "medium"}
    }
]

# Store preference memories
preference_ids = []
for pref_memory in preference_memories:
    # Create scope string
    scope_string = f"{scope.env}:{scope.agent_id}:{scope.user_id}:{scope.session_id}"

    # Merge metadata with channel information
    metadata = pref_memory["metadata"].copy()
    metadata["channel"] = "preferences"

    memory_id = memory.store(
        content=pref_memory["content"],
        scope=scope_string,
        metadata=metadata
    )
    preference_ids.append(memory_id)
    print(f"✅ Stored preference memory: {memory_id[:8]}...")

print(f"\nStored {len(preference_ids)} preference memories")

## Retrieving Memories

Now let's explore different ways to retrieve memories.

In [ ]:
# Retrieve all memories from the conversation channel
scope_string = f"{scope.env}:{scope.agent_id}:{scope.user_id}:{scope.session_id}"

# Get all memory IDs for this scope
memory_ids = memory.list(scope=scope_string)

# Retrieve memories and filter by channel
conversation_memories = []
for memory_id in memory_ids:
    memory_data = memory.retrieve(memory_id)
    if memory_data and memory_data.get('metadata', {}).get('channel') == 'conversation':
        conversation_memories.append(memory_data)

print(f"📝 Retrieved {len(conversation_memories)} conversation memories:")
for i, mem in enumerate(conversation_memories, 1):
    role = mem.get('metadata', {}).get('role', 'unknown')
    content_preview = mem['content'][:50] + ('...' if len(mem['content']) > 50 else '')
    print(f"  {i}. [{role}] {content_preview}")

In [ ]:
# Retrieve memories from context channel
scope_string = f"{scope.env}:{scope.agent_id}:{scope.user_id}:{scope.session_id}"

# Get all memory IDs for this scope
memory_ids = memory.list(scope=scope_string)

# Retrieve memories and filter by channel
context_memories = []
for memory_id in memory_ids:
    memory_data = memory.retrieve(memory_id)
    if memory_data and memory_data.get('metadata', {}).get('channel') == 'context':
        context_memories.append(memory_data)

print(f"🧠 Retrieved {len(context_memories)} context memories:")
for i, mem in enumerate(context_memories, 1):
    category = mem.get('metadata', {}).get('category', 'unknown')
    confidence = mem.get('metadata', {}).get('confidence', 'unknown')
    print(f"  {i}. [{category}] {mem['content']} (confidence: {confidence})")

In [ ]:
# Retrieve memories from preferences channel
scope_string = f"{scope.env}:{scope.agent_id}:{scope.user_id}:{scope.session_id}"

# Get all memory IDs for this scope
memory_ids = memory.list(scope=scope_string)

# Retrieve memories and filter by channel
preference_memories = []
for memory_id in memory_ids:
    memory_data = memory.retrieve(memory_id)
    if memory_data and memory_data.get('metadata', {}).get('channel') == 'preferences':
        preference_memories.append(memory_data)

print(f"⚙️ Retrieved {len(preference_memories)} preference memories:")
for i, mem in enumerate(preference_memories, 1):
    pref_type = mem.get('metadata', {}).get('type', 'unknown')
    importance = mem.get('metadata', {}).get('importance', 'unknown')
    print(f"  {i}. [{pref_type}] {mem['content']} (importance: {importance})")

## Retrieving Specific Memories

Let's retrieve specific memories by their IDs.

In [ ]:
# Retrieve a specific memory by ID
if conversation_ids:
    specific_memory_id = conversation_ids[0]
    specific_memory = memory.retrieve(specific_memory_id)

    if specific_memory:
        print(f"🎯 Retrieved specific memory (ID: {specific_memory_id[:8]}...):")
        print(f"   Content: {specific_memory['content']}")
        print(f"   Metadata: {specific_memory.get('metadata', {})}")
    else:
        print("❌ Memory not found")
else:
    print("❌ No conversation memories to retrieve")

## Memory Search Functionality

Let's explore searching memories by content.

In [ ]:
# Search for memories containing specific terms
search_terms = ["investment", "retirement", "beginner"]
scope_string = f"{scope.env}:{scope.agent_id}:{scope.user_id}:{scope.session_id}"

for term in search_terms:
    print(f"🔍 Searching for '{term}':")

    try:
        # Search returns memory IDs matching the query
        search_result_ids = memory.search(query=term, scope=scope_string)

        if search_result_ids:
            print(f"   📌 Found {len(search_result_ids)} result(s):")

            # Group results by channel
            for channel_name in channels:
                channel_results = []
                for memory_id in search_result_ids:
                    memory_data = memory.retrieve(memory_id)
                    if memory_data and memory_data.get('metadata', {}).get('channel') == channel_name:
                        channel_results.append(memory_data)

                if channel_results:
                    print(f"   📁 In '{channel_name}' channel:")
                    for result in channel_results:
                        content_preview = result['content'][:60] + ('...' if len(result['content']) > 60 else '')
                        print(f"      • {content_preview}")
        else:
            print(f"   🚫 No results found for '{term}'")

    except Exception as e:
        print(f"   ⚠️ Search error: {str(e)}")

    print()  # Empty line for readability

## Memory Statistics and Information

Let's get some statistics about our stored memories.

In [ ]:
# Get memory statistics for each channel
print("📊 Memory Statistics:")
print("=" * 50)

scope_string = f"{scope.env}:{scope.agent_id}:{scope.user_id}:{scope.session_id}"
total_memories = 0

# Get all memory IDs for this scope
memory_ids = memory.list(scope=scope_string)

# Group memories by channel
channel_memory_counts = {}
channel_memories = {}

for memory_id in memory_ids:
    memory_data = memory.retrieve(memory_id)
    if memory_data:
        channel = memory_data.get('metadata', {}).get('channel', 'unknown')
        if channel not in channel_memory_counts:
            channel_memory_counts[channel] = 0
            channel_memories[channel] = []
        channel_memory_counts[channel] += 1
        channel_memories[channel].append(memory_data)

for channel_name in channels:
    count = channel_memory_counts.get(channel_name, 0)
    total_memories += count

    print(f"📁 {channel_name.capitalize()} Channel:")
    print(f"   • Memory count: {count}")

    # Get channel config from channels dictionary
    channel_config = channels[channel_name]
    print(f"   • TTL: {channel_config.ttl_sec} seconds")

    if hasattr(channel_config, 'max_entries'):
        print(f"   • Max entries: {channel_config.max_entries}")

    if count > 0:
        # Calculate average content length
        channel_mems = channel_memories[channel_name]
        avg_length = sum(len(mem['content']) for mem in channel_mems) / len(channel_mems)
        print(f"   • Average content length: {avg_length:.1f} characters")

    print()

print(f"🎯 Total memories across all channels: {total_memories}")

## Memory Updates and Modifications

Let's demonstrate updating existing memories.

In [ ]:
# Update a preference memory
if preference_ids:
    memory_to_update = preference_ids[0]

    # Get the original memory
    original_memory = memory.retrieve(memory_to_update)

    print(f"📝 Original memory: {original_memory['content']}")

    # Update the memory by storing new content with the same ID
    updated_content = "Risk tolerance: Moderate (updated after discussion)"
    updated_metadata = {
        **original_memory.get('metadata', {}),
        "updated_at": datetime.now().isoformat(),
        "update_reason": "User provided more information"
    }

    scope_string = f"{scope.env}:{scope.agent_id}:{scope.user_id}:{scope.session_id}"

    # Store the updated memory with the same ID (this overwrites the original)
    updated_id = memory.store(
        content=updated_content,
        memory_id=memory_to_update,
        scope=scope_string,
        metadata=updated_metadata
    )

    if updated_id == memory_to_update:
        print("✅ Memory updated successfully")

        # Verify the update
        updated_memory = memory.retrieve(memory_to_update)
        print(f"📝 Updated memory: {updated_memory['content']}")
        print(f"📝 Update metadata: {updated_memory.get('metadata', {}).get('update_reason', 'N/A')}")
    else:
        print("❌ Failed to update memory")
else:
    print("❌ No preference memories to update")

## Memory Deletion

Let's demonstrate how to delete memories.

In [ ]:
# Create a temporary memory for deletion demo
scope_string = f"{scope.env}:{scope.agent_id}:{scope.user_id}:{scope.session_id}"

temp_memory_id = memory.store(
    content="This is a temporary memory for deletion demo",
    scope=scope_string,
    metadata={"temporary": True, "demo": True, "channel": "context"}
)

print(f"📝 Created temporary memory: {temp_memory_id[:8]}...")

# Verify it exists
temp_memory = memory.retrieve(temp_memory_id)

if temp_memory:
    print(f"✅ Confirmed memory exists: {temp_memory['content']}")

    # Now delete it
    deletion_success = memory.delete(temp_memory_id)

    if deletion_success:
        print("🗑️ Memory deleted successfully")

        # Verify deletion
        deleted_memory = memory.retrieve(temp_memory_id)

        if deleted_memory is None:
            print("✅ Confirmed memory no longer exists")
        else:
            print("⚠️ Memory still exists after deletion attempt")
    else:
        print("❌ Failed to delete memory")
else:
    print("❌ Temporary memory was not created properly")

## Working with Multiple Sessions

Let's demonstrate how memories are isolated by session.

In [ ]:
# Create a new scope with a different session
new_session_scope = MemoryScope(
    env=scope.env,
    agent_id=scope.agent_id,
    user_id=scope.user_id,
    session_id=str(uuid.uuid4())  # New session ID
)

print("🔄 Created new session scope:")
print(f"   Original session: {scope.session_id}")
print(f"   New session: {new_session_scope.session_id}")

# Store a memory in the new session
new_session_scope_string = f"{new_session_scope.env}:{new_session_scope.agent_id}:{new_session_scope.user_id}:{new_session_scope.session_id}"

new_session_memory_id = memory.store(
    content="This is a conversation in a new session",
    scope=new_session_scope_string,
    metadata={"session": "new", "role": "user", "channel": "conversation"}
)

print(f"📝 Stored memory in new session: {new_session_memory_id[:8]}...")

# Check memories in original session
original_scope_string = f"{scope.env}:{scope.agent_id}:{scope.user_id}:{scope.session_id}"
original_memory_ids = memory.list(scope=original_scope_string)
original_conversation_memories = []
for memory_id in original_memory_ids:
    memory_data = memory.retrieve(memory_id)
    if memory_data and memory_data.get('metadata', {}).get('channel') == 'conversation':
        original_conversation_memories.append(memory_data)

print(f"📊 Original session has {len(original_conversation_memories)} conversation memories")

# Check memories in new session
new_session_memory_ids = memory.list(scope=new_session_scope_string)
new_session_conversation_memories = []
for memory_id in new_session_memory_ids:
    memory_data = memory.retrieve(memory_id)
    if memory_data and memory_data.get('metadata', {}).get('channel') == 'conversation':
        new_session_conversation_memories.append(memory_data)

print(f"📊 New session has {len(new_session_conversation_memories)} conversation memories")

print("\n✅ Sessions are properly isolated - memories don't leak between sessions")

## Memory Export and Summary

Let's create a summary of all memories for easy review.

In [ ]:
# Create a comprehensive summary of all memories
def create_memory_summary(memory_instance, scope_obj, channels_dict):
    """Create a comprehensive summary of all memories."""
    summary = {
        "scope": scope_obj.model_dump(),
        "timestamp": datetime.now().isoformat(),
        "channels": {}
    }

    total_memories = 0
    scope_string = f"{scope_obj.env}:{scope_obj.agent_id}:{scope_obj.user_id}:{scope_obj.session_id}"

    # Get all memory IDs for this scope
    memory_ids = memory_instance.list(scope=scope_string)

    # Group memories by channel
    channel_memories_dict = {}
    for memory_id in memory_ids:
        memory_data = memory_instance.retrieve(memory_id)
        if memory_data:
            channel = memory_data.get('metadata', {}).get('channel', 'unknown')
            if channel not in channel_memories_dict:
                channel_memories_dict[channel] = []
            # Add the memory ID to the memory data for reference
            memory_data['id'] = memory_id
            channel_memories_dict[channel].append(memory_data)

    for channel_name in channels_dict:
        try:
            channel_memories = channel_memories_dict.get(channel_name, [])

            summary["channels"][channel_name] = {
                "count": len(channel_memories),
                "config": {
                    "ttl_seconds": channels_dict[channel_name].ttl_sec,
                    "description": getattr(channels_dict[channel_name], 'description', 'No description')
                },
                "memories": []
            }

            for mem in channel_memories:
                summary["channels"][channel_name]["memories"].append({
                    "id": mem.get("id", "unknown")[:8] + "...",
                    "content_length": len(mem["content"]),
                    "content_preview": mem["content"][:100] + ("..." if len(mem["content"]) > 100 else ""),
                    "metadata_keys": list(mem.get("metadata", {}).keys())
                })

            total_memories += len(channel_memories)

        except Exception as e:
            summary["channels"][channel_name] = {"error": str(e)}

    summary["total_memories"] = total_memories
    return summary

# Generate and display the summary
print("📋 Generating comprehensive memory summary...")
summary = create_memory_summary(memory, scope, channels)

print("\n📊 Memory Summary Report")
print("=" * 50)
print(f"🌍 Environment: {summary['scope']['env']}")
print(f"🤖 Agent: {summary['scope']['agent_id']}")
print(f"👤 User: {summary['scope']['user_id']}")
print(f"🔗 Session: {summary['scope']['session_id']}")
print(f"🕐 Generated: {summary['timestamp']}")
print(f"📈 Total Memories: {summary['total_memories']}")

print("\n📁 Channel Breakdown:")
for channel_name, channel_data in summary['channels'].items():
    if 'error' in channel_data:
        print(f"  ❌ {channel_name}: {channel_data['error']}")
    else:
        print(f"  📂 {channel_name.capitalize()}: {channel_data['count']} memories")
        print(f"     ⏱️ TTL: {channel_data['config']['ttl_seconds']} seconds")

        if channel_data['memories']:
            print("     📝 Recent memories:")
            for mem in channel_data['memories'][:3]:  # Show first 3
                print(f"       • {mem['id']} - {mem['content_preview']}")
        print()

## Memory System Performance Test

Let's do a quick performance test with batch operations.

In [ ]:
import time

# Performance test: Store multiple memories
print("🚀 Performance Test: Batch Memory Operations")
print("=" * 50)

# Generate test data
test_memories = []
for i in range(50):
    test_memories.append({
        "content": f"Test memory {i+1}: This is sample content for performance testing.",
        "metadata": {
            "test_id": i+1,
            "batch": "performance_test",
            "created_at": datetime.now().isoformat(),
            "channel": "context"
        }
    })

# Test storage performance
scope_string = f"{scope.env}:{scope.agent_id}:{scope.user_id}:{scope.session_id}"
start_time = time.time()
test_memory_ids = []

for test_mem in test_memories:
    memory_id = memory.store(
        content=test_mem["content"],
        scope=scope_string,
        metadata=test_mem["metadata"]
    )
    test_memory_ids.append(memory_id)

storage_time = time.time() - start_time
print(f"✅ Stored {len(test_memories)} memories in {storage_time:.3f} seconds")
print(f"📊 Average storage time: {(storage_time/len(test_memories)*1000):.2f} ms per memory")

# Test retrieval performance
start_time = time.time()
all_memory_ids = memory.list(scope=scope_string)
all_context_memories = []
for memory_id in all_memory_ids:
    memory_data = memory.retrieve(memory_id)
    if memory_data and memory_data.get('metadata', {}).get('channel') == 'context':
        all_context_memories.append(memory_data)

retrieval_time = time.time() - start_time

print(f"✅ Retrieved {len(all_context_memories)} memories in {retrieval_time:.3f} seconds")
print(f"📊 Total context memories now: {len(all_context_memories)}")

# Clean up test memories
start_time = time.time()
deleted_count = 0

for memory_id in test_memory_ids:
    if memory.delete(memory_id):
        deleted_count += 1

cleanup_time = time.time() - start_time
print(f"🗑️ Deleted {deleted_count} test memories in {cleanup_time:.3f} seconds")
if deleted_count > 0:
    print(f"📊 Average deletion time: {(cleanup_time/deleted_count*1000):.2f} ms per memory")

## Final Memory State

Let's check the final state of our memory system.

In [ ]:
# Final state check
print("🏁 Final Memory System State")
print("=" * 50)

final_stats = {}
total_final_memories = 0
scope_string = f"{scope.env}:{scope.agent_id}:{scope.user_id}:{scope.session_id}"

# Get all memory IDs for this scope
memory_ids = memory.list(scope=scope_string)

# Group memories by channel
channel_memory_counts = {}
channel_memories_final = {}

for memory_id in memory_ids:
    memory_data = memory.retrieve(memory_id)
    if memory_data:
        channel = memory_data.get('metadata', {}).get('channel', 'unknown')
        if channel not in channel_memory_counts:
            channel_memory_counts[channel] = 0
            channel_memories_final[channel] = []
        channel_memory_counts[channel] += 1
        channel_memories_final[channel].append(memory_data)

for channel_name in channels:
    try:
        channel_memories = channel_memories_final.get(channel_name, [])
        count = len(channel_memories)
        total_final_memories += count
        final_stats[channel_name] = count

        print(f"📁 {channel_name.capitalize()}: {count} memories")

        if channel_memories:
            # Show the latest memory in each channel
            latest_memory = channel_memories[-1]  # Assuming chronological order
            preview = latest_memory['content'][:60] + ('...' if len(latest_memory['content']) > 60 else '')
            print(f"   Latest: {preview}")

    except Exception as e:
        print(f"❌ Error checking {channel_name}: {str(e)}")
        final_stats[channel_name] = "Error"

print(f"\n🎯 Total memories in system: {total_final_memories}")
print(f"📊 Memory distribution: {final_stats}")
print("\n✅ Memory system demo completed successfully!")

## Summary

This notebook demonstrated the core memory management capabilities of NexaMem:

### Key Features Covered:
- **Memory Storage**: Storing memories across different channels
- **Memory Retrieval**: Getting memories by scope, channel, and ID
- **Memory Organization**: Using scopes for hierarchical organization
- **Channel Separation**: Organizing memories by type/purpose
- **Memory Updates**: Modifying existing memories
- **Memory Deletion**: Removing unwanted memories
- **Session Isolation**: Keeping memories separate between sessions
- **Search Functionality**: Finding memories by content
- **Performance Testing**: Batch operations and timing

### Memory Scope Structure:
```
{env}:{agent_id}:{user_id}:{session_id}:{channel}:{memory_id}
```

### Channel Types Used:
- **Conversation**: Main dialogue history
- **Context**: Extracted facts and information
- **Preferences**: User settings and preferences

This memory system provides a solid foundation for building AI applications that need to maintain context and learn from interactions over time.